# Synthetic ↔ real Arabic alignment, without training

Render Arabic with **arabic_reshaper + python-bidi + Pillow**, then compare the rendered images with held-out real lines from **Books A, B and C**. The checkpoint is `real_abc_balanced_optional_space_20261108`; all three books were included in its training, while these real anchors belong to its saved **test** split.

No training, optimizer, fine-tuning, calibration, vocabulary rebuilding, or threshold search occurs here. The existing word matcher and its fixed training vocabulary remain unchanged. Reference text supplies synthetic stimuli and annotation-based evaluation labels; **the matcher receives images/features only**.

This is a small diagnostic of transfer from handwriting to fonts, not evidence that real↔real alignment improved. Annotated source blocks can contain subwords: span localization here is **not certified whole-word accuracy**.

In [4]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "evaluation_utils.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the AlignmentProject checkout")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scripts.eval.synthetic_real_frozen import (
    load_frozen, generate_cases, run_cases, summarize, save_report, ROOT,
)
# Rendering dependencies are in requirements.txt: arabic-reshaper, python-bidi,
# Pillow and fonttools. This notebook also uses torch, pandas and matplotlib.
OUTPUT = ROOT / "Results/Evaluation/synthetic_real_frozen_20261109"
ANCHORS_PER_BOOK = 2
SEED = 42
FONTS = [ROOT / "Fonts/Arslan_Wessam.ttf", ROOT / "Fonts/Amiri-Regular.ttf"]
DEVICE = "auto"  # CUDA, MPS, then CPU. No downloads or pretrained reinitialization.
OUTPUT.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'arabic_reshaper'

## Frozen model and source split

Preprocessing preserves the complete line, as in the existing evaluation. Parameters and buffers are fingerprinted before and after inference. Saved split membership is mandatory. The split audit below also checks decoded image duplicates and page identities; any overlap involving test anchors stops the run.

In [ ]:
session = load_frozen(device=DEVICE)
print("Checkpoint:", session.checkpoint.name)
print("Weights folder:", session.checkpoint.parent.name)
print("SHA256:", session.checkpoint_sha256)
print("Device:", session.device, "| optional-space mode:", session.config.space_mode)
print("Saved real-line split sizes:", session.split_sizes)
print("Checkpoint trained on A, B and C; selected anchors are saved-test lines.")
print("Matcher:", session.settings["decoder"], "| vocabulary:", session.settings["word_vocabulary_path"])

In [ ]:
from scripts.eval.mask_quantitative import audit_splits
split_audit = audit_splits(session)
test_image_overlap = [r for r in split_audit["image_leakage"] if "test" in r["splits"]]
test_page_overlap = [r for r in split_audit["page_group_leakage"] if "test" in r["splits"]]
if test_image_overlap or test_page_overlap:
    raise RuntimeError("Saved test overlaps train/validation; stop before evaluation")
print("Test image overlaps:", len(test_image_overlap), "| test page overlaps:", len(test_page_overlap))
print("Other split image overlaps (reported, not hidden):", split_audit["image_leakage"])
(OUTPUT / "split_audit.json").write_text(json.dumps(split_audit, indent=2));

## Generate controlled pairs

Selection uses a seeded order within each book, before model predictions. Source annotations must match image hashes, image sizes and the normalized transcript character sequence. Out-of-bounds boxes and missing font glyphs are rejected and logged.

Each anchor/font produces five cases:

- **Full line:** render its complete supplied reference.
- **Separated spans:** two unique, spatially disjoint annotated blocks, with unrelated rendered text between them.
- **Reversed spans:** the same two blocks in opposite logical reading order; both source blocks remain correct matches.
- **Wide gaps:** render all annotated glyph groups with fixed 20 px gaps. These inserted gaps are a controlled stress test, not inferred word boundaries.
- **Unrelated:** a constructed phrase with no shared normalized 3-gram or known ≥2-letter word/group, including checks against unspaced C. These negatives are not human-reviewed semantic labels.

Ground truth consists of full-height column intervals, excluding outer margins. Full-line truth includes internal spaces; partial truth includes only the two shared block envelopes. These label policies are persisted in the manifest.

Partial blocks can start or end inside words, so a whole-word decoder may correctly reject some of them. Full-line controls preserve every source character. Wide-gap controls alter joins between annotated groups and are a font-spacing stress test, rather than a reconstruction of natural word spacing.

In [ ]:
manifest = generate_cases(session, OUTPUT, anchors_per_book=ANCHORS_PER_BOOK,
                          seed=SEED, fonts=FONTS)
print("Unique real anchors:", len(manifest["anchor_ids"]))
print("Rendered pairs:", len(manifest["cases"]), "| excluded candidates:", len(manifest["excluded"]))
display(pd.DataFrame(manifest["cases"])[["anchor_id", "book", "font", "case", "target"]])
if manifest["excluded"]:
    display(pd.DataFrame(manifest["excluded"]))

## Image-only inference and visual inspection

Inference is separated from generation/scoring: `predict_images(session, real_image, synthetic_image)` has no reference-text, annotation, or ground-truth argument. Synthetic images use a dedicated external-image extractor, while real images still require saved-test membership. No synthetic image is inserted into the saved split catalog.

The first eligible anchor in each book is displayed for both fonts and every case, independent of its score. Red overlays are predictions; white reference columns below show the expected extents. All configured pairs contribute to the quantitative tables.

In [ ]:
rows = run_cases(session, manifest, OUTPUT, display_examples=True)

## Quantitative results

IoU, Dice, precision and recall below are macro averages of the two image-side scores, then of **positive pairs only**. A missed match scores zero. Negative **pair FPR** means the fraction of unrelated pairs with any accepted region; masked-area fractions are reported separately. Empty negative masks do not inflate positive localization scores.

There are only six distinct real anchors at the defaults. Fonts and cases reuse them, so these are descriptive diagnostic values. In particular, twelve constructed negatives cannot establish a reliable 10% population FPR. No settings are selected using these results.

In [ ]:
print("Overall")
display(summarize(rows))
print("By real book")
display(summarize(rows, ("book",)))
print("By case")
display(summarize(rows, ("case",)))
print("By font")
display(summarize(rows, ("font",)))
print("By book and case")
display(summarize(rows, ("book", "case")))

## Verify and save

Fail if weights, model buffers, text directions, vocabulary, settings or evaluation mode changed. Generated PNGs, source/renderer manifest, per-pair CSVs, grouped tables and verification hashes are saved under the output directory. Notebook figures and tables remain embedded for review. Rerun all cells to regenerate this diagnostic; no training job is involved.

In [ ]:
verification = save_report(session, manifest, rows, OUTPUT)
display(pd.DataFrame([verification["checks"]]))
print("Training steps:", verification["training_steps"])
print("Saved to:", OUTPUT)
print("Interpretation: real-to-font transfer with frozen real-trained weights; not a real-real benchmark replacement.")